# ETFs — Exploratory Data Analysis

**Docker image**: `ml4t`

**Purpose**: Profile the 100-ETF candidate universe sourced from Yahoo Finance
and confirm the category coverage, history, and data-quality characteristics
that drive the ETF rotation case study.

**Learning objectives**:

- Load the ETF panel via `data.load_etfs` and inspect its canonical schema.
- Quantify per-symbol coverage and identify ETFs with shorter history.
- Check OHLC invariants and null rates across the full panel.
- Compare liquidity and price ranges across asset-class categories.

**Book reference**: §2.2 ("The Asset-Class Market Data Landscape" — ETPs).

**Prerequisites**: `data` package on `PYTHONPATH`; ETF parquet present at
`ML4T_DATA_PATH/etfs/market/`. Run `python data/etfs/market/download.py` if
missing.

In [1]:
"""ETFs — Exploratory data analysis of the multi-asset ETF universe."""

import polars as pl
from ml4t.data.etfs import ETFDataManager

from data import load_etfs
from utils.data_quality import check_ohlc_invariants, per_asset_stats
from utils.paths import REPO_ROOT

.venv/lib/python3.14/site-packages/kaleido/scopes/plotly.py:32: DeprecationWarning: 
Use of plotly.io.kaleido.scope.default_format is deprecated and support will be removed after September 2025.
Please use plotly.io.defaults.default_format instead.

  self.default_format = "png"
.venv/lib/python3.14/site-packages/kaleido/scopes/plotly.py:33: DeprecationWarning: 
Use of plotly.io.kaleido.scope.default_width is deprecated and support will be removed after September 2025.
Please use plotly.io.defaults.default_width instead.

  self.default_width = 700
.venv/lib/python3.14/site-packages/kaleido/scopes/plotly.py:34: DeprecationWarning: 
Use of plotly.io.kaleido.scope.default_height is deprecated and support will be removed after September 2025.
Please use plotly.io.defaults.default_height instead.

  self.default_height = 500
.venv/lib/python3.14/site-packages/kaleido/scopes/plotly.py:35: DeprecationWarning: 
Use of plotly.io.kaleido.scope.default_scale is deprecated and support will be rem

In [2]:
# Production defaults — Papermill injects overrides for CI
MAX_SYMBOLS = 0  # 0 = all

## 1. Load and Inspect

The ETF universe is stored as a single Parquet file containing daily OHLCV data
for 50 ETFs spanning multiple asset classes.

In [3]:
etfs = load_etfs()

print("=== ETF Dataset ===")
print(f"Shape: {etfs.shape}")
print(f"Columns: {etfs.columns}")

=== ETF Dataset ===
Shape: (470662, 7)
Columns: ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'symbol']


In [4]:
# Schema overview
print("\nSchema:")
for col, dtype in etfs.schema.items():
    print(f"  {col}: {dtype}")


Schema:
  timestamp: Date
  open: Float64
  high: Float64
  low: Float64
  close: Float64
  volume: Float64
  symbol: String


### Adjusted Prices

Yahoo Finance returns split- and dividend-adjusted OHLC. The `close` column is
the adjusted close, so returns can be computed directly without a separate
`adj_close` column.

## 2. Coverage Summary

In [5]:
# Unique symbols
symbols = etfs["symbol"].unique().sort().to_list()
print("=== Coverage ===")
print(f"Number of ETFs: {len(symbols)}")
print(f"\nSymbols: {', '.join(symbols)}")

=== Coverage ===
Number of ETFs: 100

Symbols: ACWI, ACWX, AGG, BIL, BND, BNDX, DBA, DBC, DIA, DVY, EEM, EFA, EMB, EWA, EWC, EWG, EWH, EWI, EWJ, EWL, EWN, EWP, EWQ, EWT, EWU, EWW, EWY, EWZ, EZA, FXB, FXE, FXI, FXY, GLD, GOVT, GSG, HYG, IAU, IBB, IEF, IEFA, IEMG, IJR, INDA, ITA, ITB, IVE, IVW, IWM, IYR, JNK, KRE, LQD, MCHI, MDY, MTUM, MUB, OIH, PPLT, QQQ, QUAL, RSP, SCHD, SDY, SHY, SLV, SMH, SOXX, SPY, THD, TIP, TLT, UNG, USMV, USO, UUP, VCSH, VEA, VGK, VIG, VLUE, VNQ, VTI, VTV, VUG, VWO, XBI, XLB, XLC, XLE, XLF, XLI, XLK, XLP, XLRE, XLU, XLV, XLY, XME, XRT


In [6]:
# Overall date range
date_range = etfs.select(
    [
        pl.col("timestamp").min().alias("start"),
        pl.col("timestamp").max().alias("end"),
        pl.col("timestamp").n_unique().alias("unique_dates"),
    ]
)

print(f"\nDate range: {date_range['start'][0]} to {date_range['end'][0]}")
print(f"Trading days: {date_range['unique_dates'][0]}")


Date range: 2006-01-03 to 2025-12-31
Trading days: 5031


In [7]:
symbol_stats = per_asset_stats(
    etfs,
    time_col="timestamp",
    asset_col="symbol",
    price_col="close",
    volume_col="volume",
)

full_start = date_range["start"][0]
full_end = date_range["end"][0]

partial = symbol_stats.filter(
    (pl.col("start").cast(pl.Date) != pl.lit(full_start).cast(pl.Date))
    | (pl.col("end").cast(pl.Date) != pl.lit(full_end).cast(pl.Date))
)

print(f"Symbols with full coverage: {len(symbol_stats) - len(partial)}")
print(f"Symbols with partial coverage: {len(partial)}")

Symbols with full coverage: 59
Symbols with partial coverage: 41


Most ETFs predate the 2006 start of the dataset, but a sizeable minority were
launched later — visible below as ETFs whose first observation is after
2006-01-03.

In [8]:
partial.select(["symbol", "start", "end", "rows"]).sort("start")

symbol,start,end,rows
str,date,date,u32
"""DBC""",2006-02-06,2025-12-31,5008
"""XBI""",2006-02-06,2025-12-31,5008
"""USO""",2006-04-10,2025-12-31,4964
"""SLV""",2006-04-28,2025-12-31,4951
"""VIG""",2006-05-02,2025-12-31,4949
…,…,…,…
"""VLUE""",2013-04-18,2025-12-31,3197
"""BNDX""",2013-06-04,2025-12-31,3165
"""QUAL""",2013-07-18,2025-12-31,3134


## 3. ETF Categories

Six categories cover 50 ETFs that span the major asset-class buckets used by
the rotation case study. The full universe contains 100 ETFs; the remaining
50 are kept as candidates for the universe-construction work in Chapter 6.

In [9]:
ETF_CATEGORIES = {
    "US Equity Broad": ["SPY", "QQQ", "IWM", "DIA", "VTI", "MDY", "IJR"],
    "US Sectors": ["XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY", "VNQ", "IYR"],
    "International": [
        "EFA",
        "EEM",
        "VEA",
        "VWO",
        "EWJ",
        "EWG",
        "FXI",
        "EWZ",
        "EWY",
        "EWC",
        "EWA",
        "ACWI",
    ],
    "Fixed Income": ["AGG", "BND", "TLT", "IEF", "SHY", "LQD", "HYG", "TIP", "EMB", "MUB"],
    "Commodities": ["GLD", "SLV", "USO", "UNG", "DBC", "GSG"],
    "Specialty": ["IBB", "SMH", "KRE", "OIH"],
}

print("=== ETF Categories ===")
for category, tickers in ETF_CATEGORIES.items():
    available = [t for t in tickers if t in symbols]
    print(f"  {category}: {len(available)}/{len(tickers)} ETFs")

=== ETF Categories ===
  US Equity Broad: 7/7 ETFs
  US Sectors: 11/11 ETFs
  International: 12/12 ETFs
  Fixed Income: 10/10 ETFs
  Commodities: 6/6 ETFs
  Specialty: 4/4 ETFs


## 4. Data Quality

In [10]:
# Check for nulls
null_counts = etfs.null_count()
total_nulls = null_counts.sum_horizontal()[0]
print("=== Data Quality ===")
print(f"Total null values: {total_nulls}")

=== Data Quality ===
Total null values: 0


In [11]:
# Check for zero volume days
zero_volume = etfs.filter(pl.col("volume") == 0)
print(f"Zero volume rows: {len(zero_volume)} ({100 * len(zero_volume) / len(etfs):.2f}%)")

Zero volume rows: 158 (0.03%)


In [12]:
# OHLC invariants
invariants = check_ohlc_invariants(etfs)
print("\nOHLC Invariants:")
for row in invariants.iter_rows(named=True):
    status = "[OK]" if row["valid_pct"] >= 99.99 else "[WARN]"
    print(f"  {status} {row['check']}: {row['valid_pct']:.2f}%")

# Count violations
violations = etfs.filter(
    (pl.col("high") < pl.col("low"))
    | (pl.col("high") < pl.col("open"))
    | (pl.col("high") < pl.col("close"))
)
print(f"\nTotal OHLC violations: {len(violations)}")


OHLC Invariants:
  [OK] high_gte_low: 100.00%
  [OK] high_gte_open: 100.00%
  [WARN] high_gte_close: 99.90%
  [OK] low_lte_open: 100.00%
  [WARN] low_lte_close: 99.94%
  [OK] volume_non_negative: 100.00%

Total OHLC violations: 473


Violations occur where `high < close` or `low > close` after price adjustment.
These arise from floating-point/rounding precision in the stored adjusted OHLC
values (the same cumulative ratio is applied across all four fields, but small
per-field rounding can break the strict invariants the raw quotes satisfied).
At a tenth of a percent of rows they are immaterial for return and feature
calculations but worth being aware of when computing intraday range statistics.

## 5. Volume and Liquidity Distribution

Understanding volume distributions helps identify liquidity constraints for trading.

In [13]:
# Average daily volume by ETF
volume_stats = (
    etfs.group_by("symbol")
    .agg(
        [
            pl.col("volume").mean().alias("avg_volume"),
            pl.col("volume").median().alias("median_volume"),
            pl.col("volume").std().alias("volume_std"),
        ]
    )
    .sort("avg_volume", descending=True)
)

print("=== Volume Distribution (Top 10 Most Liquid) ===")
volume_stats.head(10)

=== Volume Distribution (Top 10 Most Liquid) ===


symbol,avg_volume,median_volume,volume_std
str,f64,f64,f64
"""SPY""",1.2621e8,9.55605e7,9.1133e7
"""XLF""",7.3482e7,5.34418e7,6.7509e7
"""QQQ""",6.3756e7,4.64661e7,5.1702e7
"""EEM""",5.3074e7,4.80978e7,2.9103e7
"""IWM""",4.3071e7,3.48317e7,2.8307e7
"""XLE""",4.1028e7,3.5416e7,2.2452e7
"""FXI""",2.4401e7,2.10447e7,1.7195e7
"""XLU""",2.2636e7,2.05024e7,1.3144e7
"""EWZ""",2.0082e7,1.85157e7,1.0813e7


In [14]:
# Volume by category
print("\n=== Average Daily Volume by Category ===")
for category, tickers in ETF_CATEGORIES.items():
    category_vol = (
        etfs.filter(pl.col("symbol").is_in(tickers)).select(pl.col("volume").mean()).item()
    )
    if category_vol is not None:
        print(f"  {category}: {category_vol:,.0f} shares/day")


=== Average Daily Volume by Category ===
  US Equity Broad: 35,557,363 shares/day
  US Sectors: 20,205,419 shares/day
  International: 13,274,392 shares/day
  Fixed Income: 5,436,389 shares/day
  Commodities: 5,550,398 shares/day
  Specialty: 5,913,347 shares/day


## 6. Price Distribution

Price levels across the ETF universe span a wide range.

In [15]:
# Current price levels (most recent date)
latest = etfs.filter(pl.col("timestamp") == etfs["timestamp"].max())
price_dist = latest.select(
    [
        pl.col("close").min().alias("min_price"),
        pl.col("close").max().alias("max_price"),
        pl.col("close").median().alias("median_price"),
        pl.col("close").mean().alias("mean_price"),
    ]
)

print("=== Price Distribution (Latest Date) ===")
price_dist

=== Price Distribution (Latest Date) ===


min_price,max_price,median_price,mean_price
f64,f64,f64,f64
12.26,680.062744,86.874882,127.252057


In [16]:
# Price range by category
print("=== Price Range by Category ===")
for category, tickers in ETF_CATEGORIES.items():
    category_prices = latest.filter(pl.col("symbol").is_in(tickers))
    min_p = category_prices["close"].min()
    max_p = category_prices["close"].max()
    print(f"  {category}: ${min_p:.2f} - ${max_p:.2f}")

=== Price Range by Category ===
  US Equity Broad: $119.99 - $680.06
  US Sectors: $42.39 - $154.69
  International: $26.19 - $141.49
  Fixed Income: $73.35 - $109.91
  Commodities: $12.26 - $396.31
  Specialty: $64.44 - $360.13


## 7. Loading by Symbol or via the ml4t-data Library

`load_etfs(symbols=[...])` filters the panel to a subset; `ETFDataManager` is
the config-driven entry point used by the production download/refresh workflow
in `data/etfs/market/`.

In [17]:
spy = load_etfs(symbols=["SPY"])
print(f"SPY via loader: {spy.shape}")

SPY via loader: (5031, 7)


In [18]:
config_path = REPO_ROOT / "data" / "etfs" / "market" / "config.yaml"
etf_mgr = ETFDataManager.from_config(str(config_path))
configured = sum(len(group["symbols"]) for group in etf_mgr.config.tickers.values())
print(f"ETFDataManager loaded from {config_path}")
print(f"  Provider:           {etf_mgr.config.provider}")
print(f"  Date range:         {etf_mgr.config.start} to {etf_mgr.config.end}")
print(f"  Configured symbols: {configured} across {len(etf_mgr.config.tickers)} categories")

ETFDataManager loaded from data/etfs/market/config.yaml
  Provider:           yahoo
  Date range:         2006-01-01 to 2025-12-31
  Configured symbols: 100 across 9 categories


## Key Takeaways

1. **Pre-adjusted prices**: Yahoo Finance returns split- and dividend-adjusted
   OHLC. The `close` column is the adjusted close — return calculations need
   no further adjustment.
2. **Coverage**: 100 ETFs across daily history from 2006-01-03 to 2025-12-31
   (5,031 trading days), with 59 ETFs spanning the full window and 41 starting
   later as new products were launched.
3. **Categorization**: Six categories (US Equity Broad, US Sectors,
   International, Fixed Income, Commodities, Specialty) cover 50 of the 100
   ETFs and form the candidate set for the rotation case study; the other 50
   are reserved for the universe-construction work in Chapter 6.
4. **Mostly clean data**: Zero nulls and 473 OHLC violations
   (`high < close` or `low > close`) — about 0.1% of rows, immaterial for
   return and feature calculations.
5. **Liquidity variation**: SPY averages 126M shares/day; the Commodities
   bucket averages 5.5M — a 23× difference that matters for transaction-cost
   modeling in later chapters.

### Next Steps

- **§2.6 / `13_data_quality_framework`**: Systematic data-quality checks across
  the seven datasets.
- **§2.7 / `15_survivorship_bias_detection`**: Survivorship and selection bias
  in equity panels — a contrast to the ETF universe shown here.
- **Chapter 6**: Universe construction filters this 100-ETF candidate pool down
  to the trading universe used by the ETF rotation case study.